# Optiver 基线（LGB/XGB/CatBoost）

- 代码来源: https://www.kaggle.com/code/yuanzhezhou/baseline-lgb-xgb-and-catboost
- 飞书文档: https://dqpzque4vfe.feishu.cn/wiki/OlNPwx4uui6OflkB0D3c2iZOn2d

说明：整理并注释特征工程与推断流程，便于复现与阅读。

In [ ]:
import pandas as pd 
# generate_features：生成训练与推断用的基础特征集合（时间、价量、两两/三元组不平衡、s1/s2）
def generate_features(df):
    features = ['seconds_in_bucket', 'imbalance_buy_sell_flag',
               'imbalance_size', 'matched_size', 'bid_size', 'ask_size',
                'reference_price','far_price', 'near_price', 'ask_price', 'bid_price', 'wap',
                'imb_s1', 'imb_s2'
               ]
    
    # s1：流动性不平衡，>0 买侧更强；范围约 [-1,1]
    df['imb_s1'] = df.eval('(bid_size-ask_size)/(bid_size+ask_size)')
    # s2：成交与失衡的相对比值，描绘供需压力
    df['imb_s2'] = df.eval('(imbalance_size-matched_size)/(matched_size+imbalance_size)')
    
    prices = ['reference_price','far_price', 'near_price', 'ask_price', 'bid_price', 'wap']  # 价格集合
    
    # 两两价格不平衡度：(a-b)/(a+b)
    for i,a in enumerate(prices):
        for j,b in enumerate(prices):
            if i>j:
                df[f'{a}_{b}_imb'] = df.eval(f'({a}-{b})/({a}+{b})')
                features.append(f'{a}_{b}_imb')    
                    
    # 三元组不平衡度：(max-mid)/(mid-min)
    for i,a in enumerate(prices):
        for j,b in enumerate(prices):
            for k,c in enumerate(prices):
                if i>j and j>k:
                    max_ = df[[a,b,c]].max(axis=1)
                    min_ = df[[a,b,c]].min(axis=1)
                    mid_ = df[[a,b,c]].sum(axis=1)-min_-max_

                    df[f'{a}_{b}_{c}_imb2'] = (max_-mid_)/(mid_-min_)
                    features.append(f'{a}_{b}_{c}_imb2')
    
    return df[features]

TRAINING = False  # 是否执行训练流程（True）或仅推断加载模型（False）
if TRAINING:
    df_train = pd.read_csv('./train.csv')  # 加载官方训练数据
    df_ = generate_features(df_train)  # 生成训练特征集

## 模型与训练配置
- 使用 LGB、XGB、CatBoost 的回归配置，目标为 MAE/L1 损失
- 5 折训练并保存模型，推断阶段加载并做均值融合

In [ ]:
import lightgbm as lgb 
import xgboost as xgb 
import catboost as cbt 
import numpy as np 
import joblib 
import os 



model_path ='./model'  # 模型文件存储/加载路径

N_fold = 5  # 折数：用于交叉验证或分折训练

if TRAINING:
    X = df_.values  # 训练特征矩阵
    Y = df_train['target'].values  # 目标序列

    X = X[np.isfinite(Y)]  # 过滤目标为非有限值的样本
    Y = Y[np.isfinite(Y)]

    index = np.arange(len(X))  # 简易折分索引：index%N_fold==i 为第 i 折

models = []  # 训练或加载后的模型列表

def train(model_dict, modelname='lgb'):
    if TRAINING:
        model = model_dict[modelname]  # 选择模型
        model.fit(X[index%N_fold!=i], Y[index%N_fold!=i], 
                    eval_set=[(X[index%N_fold==i], Y[index%N_fold==i])], 
                    verbose=10, 
                    early_stopping_rounds=100  # 早停：验证集100轮无提升
                    )
        models.append(model)
        joblib.dump(model, './models/{modelname}_{i}.model')  # 保存单折模型
    else:
        models.append(joblib.load(f'{model_path}/{modelname}_{i}.model'))  # 推断时加载模型
    return 

model_dict = {
    'lgb': lgb.LGBMRegressor(objective='regression_l1', n_estimators=500),  # LGB：L1回归
    'xgb': xgb.XGBRegressor(tree_method='hist', objective='reg:absoluteerror', n_estimators=500),  # XGB：MAE
    'cbt': cbt.CatBoostRegressor(objective='MAE', iterations=3000),  # CatBoost：MAE

}

for i in range(N_fold):
    train(model_dict, 'lgb')  # 训练/加载 LGB 第 i 折
#     train(model_dict, 'xgb')
    train(model_dict, 'cbt')  # 训练/加载 CatBoost 第 i 折


## 竞赛环境与推断
- 使用 `optiver2023` 提供的评测环境进行迭代推断
- 每次窗口内计算特征并对目标进行预测后提交

In [3]:
import optiver2023
env = optiver2023.make_env()  # 竞赛评测环境
iter_test = env.iter_test()  # 迭代式测试数据流接口

In [4]:
counter = 0  # 流式推断计数器
for (test, revealed_targets, sample_prediction) in iter_test:
    feat = generate_features(test)  # 逐批生成特征
    
    # 多模型均值融合：LGB/CatBoost（与XGB可选），提升稳健性
    sample_prediction['target'] = np.mean([model.predict(feat) for model in models], 0)
    env.predict(sample_prediction)
    counter += 1  # 统计迭代步数

This version of the API is not optimized and should not be used to estimate the runtime of your code on the hidden test set.
